# Rubik's Cube View Sweep

This notebook is for unattended Colab runs.

It keeps the working official-like-but-memory-safe `128x128` Fourier setup, then increases the number of **individual target views** one by one so you can see how much the current setup can handle before it degrades or runs out of memory.


## Before You Run

- Use a **GPU** runtime.
- This notebook is designed to run unattended for several hours.
- If this is your first install in the runtime, restart after the install cell and rerun from the top.
- Each sweep step writes outputs and metadata under `output/colab-face-count-sweep`.
- At the end, the notebook writes a `sweep-summary.json` file and can export the whole sweep as a zip.


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

OUR_REPO_URL = 'https://github.com/netalondon/rubiks-diffusion-illusion.git'
OUR_REPO_DIR = Path('/content/rubiks-diffusion-illusion')
OFFICIAL_REPO_URL = 'https://github.com/RyannDaGreat/Diffusion-Illusions.git'
OFFICIAL_REPO_DIR = Path('/content/Diffusion-Illusions')

for repo_url, repo_dir in [
    (OUR_REPO_URL, OUR_REPO_DIR),
    (OFFICIAL_REPO_URL, OFFICIAL_REPO_DIR),
]:
    if repo_dir.exists():
        print(f'Reusing existing repo at {repo_dir}')
        subprocess.check_call(['git', '-C', str(repo_dir), 'pull', '--ff-only'])
    else:
        subprocess.check_call(['git', 'clone', repo_url, str(repo_dir)])

print('Python:', sys.version)
print('Platform:', platform.platform())
print('In Colab runtime:', 'COLAB_RELEASE_TAG' in os.environ)
print('Our repo:', OUR_REPO_DIR)
print('Official repo:', OFFICIAL_REPO_DIR)


In [ ]:
import subprocess
import sys

third_party_packages = [
    'diffusers',
    'transformers',
    'rp',
    'easydict',
    'einops',
    'icecream',
    'imageio',
    'opencv-python',
    'pandas',
    'scipy',
    'tensorboard',
    'tensorboardX',
    'tqdm',
    'py3nvml',
]

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    *third_party_packages,
])
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-r',
    str(OUR_REPO_DIR / 'requirements.txt'),
])

print('Installed the Colab-compatible dependency set for the Rubik + Diffusion Illusions sweep.')
print('If this was the first install in this runtime, restart now, then rerun from the repo bootstrap cell at the top.')


## Imports And Setup

In [ ]:
import gc
import json
import shutil
import sys
import time
import traceback
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

OUR_REPO_DIR = globals().get('OUR_REPO_DIR', Path('/content/rubiks-diffusion-illusion'))
OFFICIAL_REPO_DIR = globals().get('OFFICIAL_REPO_DIR', Path('/content/Diffusion-Illusions'))

for repo_dir in [OUR_REPO_DIR, OFFICIAL_REPO_DIR]:
    if not repo_dir.exists():
        raise FileNotFoundError(f'Missing expected repo directory: {repo_dir}. Run the repo bootstrap cell first.')

if str(OUR_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OUR_REPO_DIR))
if str(OFFICIAL_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_REPO_DIR))

import matplotlib.pyplot as plt
import numpy as np
import rp
import torch
import torch.nn as nn

import source.stable_diffusion as sd
from python_bridge import (
    batch_to_pil_face_dict,
    clone_rendered_to_cpu,
    load_source_faces,
    load_spec,
    render_all_arrangements,
    render_all_arrangements_torch,
    save_rendered_faces,
    stack_source_face_tensors,
    tensor_to_pil,
)
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel

gpu = rp.select_torch_device()
if str(gpu) == 'cpu':
    raise RuntimeError('This sweep should run on a GPU runtime. Switch Colab to GPU and reconnect.')

if 's' not in globals():
    model_name = 'runwayml/stable-diffusion-v1-5'
    s = sd.StableDiffusion(gpu, model_name)

device = s.device
print('Stable Diffusion device:', device)


In [ ]:
SPEC_PATH = OUR_REPO_DIR / 'public' / 'generated' / 'rubiks-illusion-spec.json'
SOURCE_DIR = OUR_REPO_DIR / 'src' / 'assets' / 'face-art'

spec = load_spec(SPEC_PATH)
face_order = list(spec['primeImages'])
face_to_index = {face_id: index for index, face_id in enumerate(face_order)}

source_faces_pil = load_source_faces(SOURCE_DIR, face_order)
source_batch = stack_source_face_tensors(source_faces_pil, face_order, device=device)

pil_rendered = render_all_arrangements(spec, source_faces_pil)
torch_rendered = render_all_arrangements_torch(spec, source_batch, face_to_index)

max_diff = 0.0
for arrangement_name in pil_rendered:
    for face_id, pil_image in pil_rendered[arrangement_name].items():
        pil_tensor = stack_source_face_tensors({face_id: pil_image}, [face_id], device=device)[0]
        diff = (pil_tensor - torch_rendered[arrangement_name][face_id]).abs().max().item()
        max_diff = max(max_diff, diff)

print('Face order:', face_order)
print('Arrangement names:', list(spec['arrangements'].keys()))
print(f'Max abs difference between PIL and torch Rubik renders: {max_diff:.6f}')


## Sweep Config

This sweep uses the same prompt on every solved target view and the same prompt on every scrambled target view.

It is anchored on the currently proven seed pair (`solved:R` and `scrambled:U`), then starts the unattended sweep at `3` views by adding one new target view at a time until all `12` solved/scrambled views are included.


In [ ]:
VIEW_GROWTH_ORDER = [
    ('solved', 'R'),
    ('scrambled', 'U'),
    ('solved', 'U'),
    ('scrambled', 'R'),
    ('solved', 'L'),
    ('scrambled', 'L'),
    ('solved', 'F'),
    ('scrambled', 'F'),
    ('solved', 'D'),
    ('scrambled', 'D'),
    ('solved', 'B'),
    ('scrambled', 'B'),
]
VIEW_COUNTS = list(range(3, len(VIEW_GROWTH_ORDER) + 1))
SOLVED_PROMPT = 'a watercolor drawing of a cute cat'
SCRAMBLED_PROMPT = 'a watercolor drawing of a cute dog'
NEGATIVE_PROMPT = 'blurry, noisy, muddy colors, text, watermark, low quality, collage, multiple animals, cropped face'

PARAMETERIZATION_MODE = 'official_fourier_128'
LEARNABLE_SIZE = 128
FOURIER_HIDDEN_DIM = 128
FOURIER_NUM_FEATURES = 128
FOURIER_SCALE = 10

ITERATIONS_PER_VIEW = 1000
DISPLAY_INTERVAL = 50
LEARNING_RATE = 1e-4
GUIDANCE_SCALE = 100
NOISE_COEF = 0.10
OPTIMIZER_NAME = 'SGD'
CONTINUE_AFTER_FAILURE = True

s.max_step = 990
s.min_step = 10

print('View growth order:', [f'{arrangement}:{face}' for arrangement, face in VIEW_GROWTH_ORDER])
print('View counts:', VIEW_COUNTS)
print('Solved prompt:', SOLVED_PROMPT)
print('Scrambled prompt:', SCRAMBLED_PROMPT)
print('Parameterization mode:', PARAMETERIZATION_MODE)
print('Iterations per view:', ITERATIONS_PER_VIEW)
print('Display interval:', DISPLAY_INTERVAL)
print('Learning rate:', LEARNING_RATE)
print('Guidance scale:', GUIDANCE_SCALE)
print('Noise coefficient:', NOISE_COEF)
print('Min/max step:', (s.min_step, s.max_step))


In [ ]:
class RubiksLearnableSourceFacesFourier(nn.Module):
    def __init__(self, face_ids, size, hidden_dim=128, num_features=128, scale=10):
        super().__init__()
        self.face_ids = list(face_ids)
        self.learnable_images = nn.ModuleList([
            LearnableImageFourier(
                height=size,
                width=size,
                hidden_dim=hidden_dim,
                num_features=num_features,
                scale=scale,
            )
            for _ in self.face_ids
        ])

    def forward(self):
        return torch.stack([image() for image in self.learnable_images])


def build_train_views(selected_views):
    views = []
    for arrangement, face in selected_views:
        prompt = SOLVED_PROMPT if arrangement == 'solved' else SCRAMBLED_PROMPT
        views.append({
            'name': f'{arrangement}:{face}',
            'arrangement': arrangement,
            'face': face,
            'prompt': prompt,
            'weight': 1.0,
            'label': NegativeLabel(prompt, NEGATIVE_PROMPT),
        })
    return views


def show_face_dict(face_dict, title, face_ids):
    cols = min(3, max(1, len(face_ids)))
    rows = (len(face_ids) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
    axes = np.asarray(axes, dtype=object).reshape(rows, cols)
    fig.suptitle(title, fontsize=14)

    for axis in axes.flat:
        axis.axis('off')

    for index, face_id in enumerate(face_ids):
        row = index // cols
        col = index % cols
        axis = axes[row, col]
        axis.imshow(np.asarray(face_dict[face_id]))
        axis.set_title(face_id)
        axis.axis('off')

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def show_training_views(rendered_views, train_views, title):
    items = [
        (view['name'], rendered_views[view['arrangement']][view['face']])
        for view in train_views
    ]
    cols = min(4, max(1, len(items)))
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
    axes = np.asarray(axes, dtype=object).reshape(rows, cols)
    fig.suptitle(title, fontsize=14)

    for axis in axes.flat:
        axis.axis('off')

    for index, (label, image_tensor) in enumerate(items):
        row = index // cols
        col = index % cols
        axis = axes[row, col]
        axis.imshow(np.asarray(tensor_to_pil(image_tensor)))
        axis.set_title(label)
        axis.axis('off')

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def make_run_dirs(view_count):
    sweep_root = OUR_REPO_DIR / 'output' / 'colab-view-count-sweep'
    count_slug = f'{view_count}-views'
    output_root = sweep_root / count_slug
    run_timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
    run_root = OUR_REPO_DIR / 'output' / 'colab-runs' / f'{run_timestamp}-view-sweep-{count_slug}'
    return sweep_root, output_root, run_root


def save_run_outputs(output_root, run_root, final_source_faces, final_rendered_cpu, metadata):
    source_output_dir = output_root / 'source-faces'
    render_output_dir = output_root / 'rendered'
    run_source_dir = run_root / 'results' / 'source-faces'
    run_render_dir = run_root / 'results' / 'rendered'
    notebook_copy_path = run_root / 'notebook-source.ipynb'
    metadata_path = run_root / 'metadata.json'

    for directory in [source_output_dir, run_source_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    for face_id, image in final_source_faces.items():
        image.save(source_output_dir / f'{face_id}.png')
        image.save(run_source_dir / f'{face_id}.png')

    save_rendered_faces(
        {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['solved'].items()},
        render_output_dir / 'solved',
    )
    save_rendered_faces(
        {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['scrambled'].items()},
        render_output_dir / 'scrambled',
    )
    save_rendered_faces(
        {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['solved'].items()},
        run_render_dir / 'solved',
    )
    save_rendered_faces(
        {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['scrambled'].items()},
        run_render_dir / 'scrambled',
    )

    run_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(OUR_REPO_DIR / 'experiments' / 'local-face-sweep' / 'rubiks_colab_face_sweep.ipynb', notebook_copy_path)
    metadata_path.write_text(json.dumps(metadata, indent=2) + '\n')

    return {
        'source_output_dir': str(source_output_dir),
        'render_output_dir': str(render_output_dir),
        'run_root': str(run_root),
        'metadata_path': str(metadata_path),
    }


def run_single_view_count(view_count):
    selected_views = VIEW_GROWTH_ORDER[:view_count]
    train_views = build_train_views(selected_views)

    sweep_root, output_root, run_root = make_run_dirs(view_count)
    output_root.mkdir(parents=True, exist_ok=True)

    learnable_faces = RubiksLearnableSourceFacesFourier(
        face_order,
        LEARNABLE_SIZE,
        hidden_dim=FOURIER_HIDDEN_DIM,
        num_features=FOURIER_NUM_FEATURES,
        scale=FOURIER_SCALE,
    ).to(device)
    optim = torch.optim.SGD(learnable_faces.parameters(), lr=LEARNING_RATE)

    def get_current_state():
        current_source_batch = learnable_faces()
        current_rendered = render_all_arrangements_torch(spec, current_source_batch, face_to_index)
        return current_source_batch, current_rendered

    num_iter = view_count * ITERATIONS_PER_VIEW

    start_time = time.time()
    print('\n' + '=' * 80)
    print(f'Starting sweep run for {view_count} views:', [f'{arr}:{face}' for arr, face in selected_views])
    print('Train views:', [view['name'] for view in train_views])
    print('Iterations for this run:', num_iter)

    _, initial_rendered = get_current_state()
    show_training_views(clone_rendered_to_cpu(initial_rendered), train_views, f'Initial render for {view_count} views')

    last_sampled_names = []
    for iter_num in range(num_iter):
        optim.zero_grad(set_to_none=True)
        current_source_batch, current_rendered = get_current_state()

        sampled_views = list(rp.random_batch(train_views, batch_size=1))
        last_sampled_names = [view['name'] for view in sampled_views]
        for view in sampled_views:
            current_view = current_rendered[view['arrangement']][view['face']][None]
            s.train_step(
                view['label'].embedding,
                current_view,
                noise_coef=NOISE_COEF * view['weight'],
                guidance_scale=GUIDANCE_SCALE,
            )

        optim.step()

        if (iter_num + 1) % DISPLAY_INTERVAL == 0 or iter_num == 0:
            print(f'Completed iteration {iter_num + 1}/{num_iter} | sampled={last_sampled_names}')
            _, preview_rendered = get_current_state()
            show_training_views(
                clone_rendered_to_cpu(preview_rendered),
                train_views,
                f'{view_count} views preview at iter {iter_num + 1}',
            )

    final_source_batch, final_rendered = get_current_state()
    final_source_faces = batch_to_pil_face_dict(final_source_batch.detach().cpu(), face_order)
    final_rendered_cpu = clone_rendered_to_cpu(final_rendered)

    metadata = {
        'status': 'success',
        'timestamp_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ'),
        'view_count': view_count,
        'selected_views': [f'{arrangement}:{face}' for arrangement, face in selected_views],
        'parameterization_mode': PARAMETERIZATION_MODE,
        'solved_prompt': SOLVED_PROMPT,
        'scrambled_prompt': SCRAMBLED_PROMPT,
        'negative_prompt': NEGATIVE_PROMPT,
        'train_views': [
            {
                'name': view['name'],
                'arrangement': view['arrangement'],
                'face': view['face'],
                'prompt': view['prompt'],
                'weight': view['weight'],
            }
            for view in train_views
        ],
        'hyperparameters': {
            'iterations_per_view': ITERATIONS_PER_VIEW,
            'num_iter': num_iter,
            'display_interval': DISPLAY_INTERVAL,
            'learnable_size': LEARNABLE_SIZE,
            'fourier_hidden_dim': FOURIER_HIDDEN_DIM,
            'fourier_num_features': FOURIER_NUM_FEATURES,
            'fourier_scale': FOURIER_SCALE,
            'learning_rate': LEARNING_RATE,
            'optimizer_name': OPTIMIZER_NAME,
            'guidance_scale': GUIDANCE_SCALE,
            'noise_coef': NOISE_COEF,
            'min_step': s.min_step,
            'max_step': s.max_step,
        },
        'elapsed_seconds': round(time.time() - start_time, 2),
        'last_sampled_views': last_sampled_names,
    }
    saved_paths = save_run_outputs(output_root, run_root, final_source_faces, final_rendered_cpu, metadata)
    metadata['saved_paths'] = saved_paths
    (run_root / 'metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')

    print(f'Completed {view_count}-view run in {metadata["elapsed_seconds"]:.1f}s')
    print('Saved run root:', saved_paths['run_root'])

    return metadata


def cleanup_after_run():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


In [ ]:
SWEEP_ROOT = OUR_REPO_DIR / 'output' / 'colab-view-count-sweep'
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
SWEEP_SUMMARY_PATH = SWEEP_ROOT / 'sweep-summary.json'

sweep_results = []

for view_count in VIEW_COUNTS:
    try:
        result = run_single_view_count(view_count)
    except Exception as error:
        result = {
            'status': 'failed',
            'timestamp_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ'),
            'view_count': view_count,
            'selected_views': [f'{arr}:{face}' for arr, face in VIEW_GROWTH_ORDER[:view_count]],
            'error_type': type(error).__name__,
            'error_message': str(error),
            'traceback': traceback.format_exc(),
        }
        print(f'Run failed for {view_count} views: {type(error).__name__}: {error}')
        if not CONTINUE_AFTER_FAILURE:
            sweep_results.append(result)
            cleanup_after_run()
            break
    sweep_results.append(result)
    SWEEP_SUMMARY_PATH.write_text(json.dumps(sweep_results, indent=2) + '\n')
    cleanup_after_run()

print('\nSweep finished. Summary saved to:', SWEEP_SUMMARY_PATH)
for result in sweep_results:
    line = f"{result['view_count']} views -> {result['status']}"
    if result['status'] == 'success':
        line += f" ({result['elapsed_seconds']}s)"
    else:
        line += f" ({result['error_type']})"
    print(line)


In [ ]:
import shutil

ARCHIVE_BASE = SWEEP_ROOT.parent / 'colab-view-count-sweep-export'
ARCHIVE_PATH = shutil.make_archive(str(ARCHIVE_BASE), 'zip', root_dir=SWEEP_ROOT)
print('Created export archive:', ARCHIVE_PATH)

try:
    from google.colab import files
    files.download(ARCHIVE_PATH)
    print('Started download via google.colab.files.download')
except Exception as error:
    print('Automatic download was not available in this runtime:', repr(error))
    print('The zip file is still available inside Colab at:', ARCHIVE_PATH)
    print('You can download it manually from the Colab file browser if needed.')


## What Success Looks Like

This notebook is a success if all of these are true:

- it runs through view counts `3` to `12` without manual intervention, with more total steps allocated to larger runs
- each run writes a timestamped snapshot under `output/colab-runs`
- the running summary in `output/colab-view-count-sweep/sweep-summary.json` makes it obvious where quality or stability starts to break
- the final zip export gives you one bundle you can inspect the next morning
